# MF Top-N 순위 진단 — Video_Games

05에서 선택한 설정으로 MF를 **한 번만** 다시 학습한다. 05의 하이퍼파라미터 탐색은 반복하지 않는다.
프로토콜 ③의 동일한 사용자 표본에서 MF 상위 10개 추천 상품과 테스트 정답 상품의 학습 평점 수를 비교한다.
또한 전체 MF 순위와, 같은 모델에서 잠재요인 점수만 뺀 순위가 얼마나 겹치는지 확인한다.

예상 시간: 학습 약 4~5분 + 순위 진단. 결과는 Drive의 `results/mf/`에 저장한다.
이 노트북은 05·08·09의 기존 파일을 덮어쓰지 않는다.


In [1]:
# 0. 설정과 데이터
import os, sys, json, subprocess, time
from pathlib import Path
try:
    from google.colab import drive
except ImportError:
    REPO = os.environ.get('TEAM10_REPO', os.path.abspath('..'))
else:
    drive.mount('/content/drive')
    REPO = '/content/team10-recsys'
    if os.path.isdir(os.path.join(REPO, '.git')):
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone',
                        'https://github.com/qhfl3644/team10-recsys.git', REPO], check=True)
sys.path.insert(0, REPO)

DS = 'Video_Games'
relative_data_file = Path('data/processed') / DS / 'user_map.parquet'
root_hint = os.environ.get('TEAM10_ROOT')
roots = []
if root_hint and (Path(root_hint) / relative_data_file).is_file():
    roots.append(Path(root_hint))
default_root = Path('/content/drive/MyDrive/Team10_RecSys')
if (default_root / relative_data_file).is_file():
    roots.append(default_root)
if not roots:
    # MyDrive와 공유 드라이브의 최상위·두 단계 하위에서 팀 폴더를 찾는다.
    for base in (Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives')):
        if not base.is_dir():
            continue
        level = [base]
        for depth in range(3):
            next_level = []
            for folder in level:
                if (folder / relative_data_file).is_file():
                    roots.append(folder)
                if depth < 2:
                    try:
                        next_level.extend(p for p in folder.iterdir() if p.is_dir())
                    except OSError:
                        pass
            level = next_level
roots = list(dict.fromkeys(str(p.resolve()) for p in roots))
if len(roots) != 1:
    raise FileNotFoundError(
        f'{DS}/user_map.parquet를 마운트된 Drive에서 찾지 못했거나 위치가 여러 곳입니다. '
        f'발견한 데이터 폴더: {roots}. '
        'Google Drive에서 Team10_RecSys 폴더를 내 드라이브에 바로가기 추가한 뒤 '
        '첫 셀을 다시 실행하세요. 경로를 알고 있다면 config를 import하기 전에 '
        "os.environ['TEAM10_ROOT'] = '실제 Team10_RecSys 경로'를 지정하세요."
    )
os.environ['TEAM10_ROOT'] = roots[0]
print('데이터 폴더:', roots[0])

import numpy as np
import pandas as pd
from config import RESULT_DIR, SEED
from src.eval import metrics as M
from src.eval.ranking import target_rank_full
from src.models.mf import MF

data = M.load_data(DS)
p3_users = M.load_p3_users(DS)
targets = data.test.set_index('uidx')['iidx'].reindex(p3_users)
if targets.isna().any():
    raise ValueError('프로토콜 ③ 사용자 중 test 정답 상품이 없는 사용자가 있습니다')
targets = targets.to_numpy(dtype=np.int64)
cfg_path = f'{RESULT_DIR}/mf/{DS}_mf_config.json'
with open(cfg_path, encoding='utf-8') as f:
    cfg = json.load(f)
print(f'{DS}: 사용자 {data.n_users:,}명, 상품 {data.n_items:,}개, 프로토콜 ③ 표본 {len(p3_users):,}명')
print(f'05 설정: k={cfg["k"]}, lr={cfg["lr"]}, reg={cfg["reg"]}')


Mounted at /content/drive
데이터 폴더: /content/drive/.shortcut-targets-by-id/1bVHT2f3FczQeEHy_Z3dsGhlCkuyCTCST/Team10_RecSys
Video_Games: 사용자 94,762명, 상품 25,612개, 프로토콜 ③ 표본 10,000명
05 설정: k=16, lr=0.01, reg=0.1


In [2]:
# 1. 05에서 선택한 MF 설정으로 한 번만 재학습
t0 = time.time()
mf = MF(k=int(cfg['k']), lr=float(cfg['lr']), reg=float(cfg['reg']),
        epochs=60, patience=5, seed=SEED, verbose=False)
mf.fit(data.train, data.n_users, data.n_items, valid=data.valid)
test_pred = mf.predict(data.test.uidx.to_numpy(), data.test.iidx.to_numpy())
test_rmse = M.rmse(data.test.rating.to_numpy(), test_pred)
print(f'재학습 완료: best epoch {mf.best_epoch}, test RMSE {test_rmse:.4f}, '
      f'소요 {time.time()-t0:.0f}초')
if 'test_rmse' in cfg and abs(test_rmse - float(cfg['test_rmse'])) > 0.002:
    raise RuntimeError(f'05 저장 결과({cfg["test_rmse"]:.4f})와 재학습 RMSE가 다릅니다. '
                       '데이터·코드 버전을 확인하세요.')


재학습 완료: best epoch 22, test RMSE 1.1714, 소요 341초


In [3]:
# 2. 프로토콜 ③의 추천 상품 인기도와 편향 기여 분석
K = 10
CHUNK = 128
rng_full = np.random.default_rng([SEED, 3])
rng_bias = np.random.default_rng([SEED, 3])
rng_top = np.random.default_rng([SEED, 505])
empty = np.empty(0, dtype=np.int64)

def topk_random_ties(scores, k, rng):
    if np.isfinite(scores).sum() < k:
        raise ValueError('추천 가능한 상품이 10개 미만입니다')
    cutoff = np.partition(scores, -k)[-k]
    higher = np.flatnonzero(scores > cutoff)
    tied = np.flatnonzero(scores == cutoff)
    rest = rng.choice(tied, size=k-len(higher), replace=False)
    return np.concatenate((higher, rest))

n = len(p3_users)
top10 = np.empty((n, K), dtype=np.int32)
rank_full = np.empty(n, dtype=np.int32)
rank_bias = np.empty(n, dtype=np.int32)
overlap = np.empty(n, dtype=np.float64)
top_abs_bias = np.empty(n, dtype=np.float64)
top_abs_factor = np.empty(n, dtype=np.float64)
t0 = time.time()
for start in range(0, n, CHUNK):
    uu = p3_users[start:start+CHUNK]
    scores_chunk = mf.score_rows(uu)
    for j, u in enumerate(uu):
        row = start + j
        target = int(targets[row])
        excluded = data.rated_except(int(u), target)
        full_scores = scores_chunk[j].copy()
        full_scores[excluded] = -np.inf
        bias_scores = mf.bi.copy()  # mu+b_u는 상품 순위에 영향을 주지 않는 상수
        bias_scores[excluded] = -np.inf

        # 제외 상품을 이미 -inf로 바꿨으므로 target_rank_full에는 빈 제외 목록을 전달
        rank_full[row] = target_rank_full(full_scores, target, empty, rng_full)
        rank_bias[row] = target_rank_full(bias_scores, target, empty, rng_bias)
        rec = topk_random_ties(full_scores, K, rng_top)
        rec_bias = topk_random_ties(bias_scores, K, rng_top)
        top10[row] = rec
        overlap[row] = len(np.intersect1d(rec, rec_bias)) / K
        top_abs_bias[row] = np.abs(mf.bi[rec]).mean()
        top_abs_factor[row] = np.abs(mf.P[int(u)] @ mf.Q[rec].T).mean()
    if start == 0 or (start + CHUNK) % 2048 == 0 or start + CHUNK >= n:
        print(f'{min(start+CHUNK, n):,}/{n:,}명 분석 완료 ({time.time()-t0:.0f}초)')

rec_count = data.item_count[top10]
target_count = data.item_count[targets]
per_user = pd.DataFrame({
    'uidx': p3_users,
    'target_iidx': targets,
    'target_train_count': target_count,
    'target_rank_mf': rank_full,
    'target_rank_bias_score': rank_bias,
    'mf_hit10': rank_full <= K,
    'recommended_train_count_mean': rec_count.mean(axis=1),
    'recommended_train_count_median': np.median(rec_count, axis=1),
    'recommended_count_le5_share': (rec_count <= 5).mean(axis=1),
    'top10_overlap_with_bias_score': overlap,
    'top10_abs_item_bias_mean': top_abs_bias,
    'top10_abs_factor_mean': top_abs_factor,
})
summary = pd.DataFrame([{
    'dataset': DS,
    'n_p3_users': n,
    'n_items': data.n_items,
    'mf_k': mf.k,
    'mf_best_epoch': mf.best_epoch,
    'mf_test_rmse': test_rmse,
    'mf_hr10_p3': float((rank_full <= K).mean()),
    'bias_score_hr10_p3': float((rank_bias <= K).mean()),
    'uniform_random_hr10_approx': K / data.n_items,
    'mf_target_rank_median': float(np.median(rank_full)),
    'recommended_train_count_mean': float(rec_count.mean()),
    'recommended_train_count_median': float(np.median(rec_count)),
    'target_train_count_mean': float(target_count.mean()),
    'target_train_count_median': float(np.median(target_count)),
    'recommended_count_le5_share': float((rec_count <= 5).mean()),
    'target_count_le5_share': float((target_count <= 5).mean()),
    'recommended_unseen_share': float((rec_count == 0).mean()),
    'top10_overlap_with_bias_score_mean': float(overlap.mean()),
    'top10_abs_item_bias_mean': float(top_abs_bias.mean()),
    'top10_abs_factor_mean': float(top_abs_factor.mean()),
}])
exposure = np.bincount(top10.ravel(), minlength=data.n_items)
items = pd.DataFrame({'iidx': np.arange(data.n_items),
                      'train_count': data.item_count,
                      'item_bias': mf.bi,
                      'top10_exposures': exposure})
items = items[items.top10_exposures > 0].sort_values('top10_exposures', ascending=False)
print('\n진단 요약')
print(summary.T.to_string(header=False))
print('\n상위 노출 상품 10개')
print(items.head(10).to_string(index=False))


128/10,000명 분석 완료 (0초)
2,048/10,000명 분석 완료 (2초)
4,096/10,000명 분석 완료 (4초)
6,144/10,000명 분석 완료 (6초)
8,192/10,000명 분석 완료 (8초)
10,000/10,000명 분석 완료 (10초)

진단 요약
dataset                             Video_Games
n_p3_users                                10000
n_items                                   25612
mf_k                                         16
mf_best_epoch                                22
mf_test_rmse                           1.171369
mf_hr10_p3                               0.0002
bias_score_hr10_p3                       0.0002
uniform_random_hr10_approx              0.00039
mf_target_rank_median                   11615.0
recommended_train_count_mean           14.93774
recommended_train_count_median             13.0
target_train_count_mean                155.3168
target_train_count_median                  49.0
recommended_count_le5_share                 0.0
target_count_le5_share                    0.108
recommended_unseen_share                    0.0
top10_overlap_with_bias_sco

In [4]:
# 3. Drive에 근거 자료 저장
out_dir = f'{RESULT_DIR}/mf'
os.makedirs(out_dir, exist_ok=True)
summary_path = f'{out_dir}/{DS}_mf_topn_diagnostic_summary.csv'
users_path = f'{out_dir}/{DS}_mf_topn_diagnostic_users.parquet'
items_path = f'{out_dir}/{DS}_mf_topn_diagnostic_items.csv'
summary.to_csv(summary_path, index=False)
per_user.to_parquet(users_path, index=False)
items.to_csv(items_path, index=False)
print('저장 완료:')
print(summary_path)
print(users_path)
print(items_path)


저장 완료:
/content/drive/.shortcut-targets-by-id/1bVHT2f3FczQeEHy_Z3dsGhlCkuyCTCST/Team10_RecSys/results/mf/Video_Games_mf_topn_diagnostic_summary.csv
/content/drive/.shortcut-targets-by-id/1bVHT2f3FczQeEHy_Z3dsGhlCkuyCTCST/Team10_RecSys/results/mf/Video_Games_mf_topn_diagnostic_users.parquet
/content/drive/.shortcut-targets-by-id/1bVHT2f3FczQeEHy_Z3dsGhlCkuyCTCST/Team10_RecSys/results/mf/Video_Games_mf_topn_diagnostic_items.csv


## 보고서 작성 시 확인할 점

- MF의 프로토콜 ③ HR@10이 기존 05/09의 **0.0002**와 비슷한지 먼저 확인한다.
- 1만 명 표본에서 0.0002는 적중 약 2명이다. 균등 무작위 기대치 약 0.00039와의
  차이는 적중 약 2명 수준이므로, 통계적으로 무작위보다 나쁘다고 단정하지 않는다.
- 추천 상위 10개의 학습 평점 수와 정답 상품의 학습 평점 수를 비교한다. 추천 쪽이 적다면
  저인기도 상품에 순위가 쏠렸다는 가설을 뒷받침한다. 반대라면 가설을 수정한다.
- `top10_overlap_with_bias_score_mean`이 높을수록 상품 편향이 추천 순위를 많이 결정했다는 근거다.
  이 값만으로 원인을 단정하지 말고 평점 수·편향 크기와 함께 설명한다.
- `mf_hr10_p3`와 `bias_score_hr10_p3`는 적중 건수가 매우 적을 수 있다.
  소수점 네 자리 차이만으로 두 순위의 우열을 단정하지 않는다.

실행 출력과 세 결과 파일을 보관한 뒤, 5.4 MF 랭킹 원인 문단에는 **실제로 나온 값**을 적는다.
